## Similarity Exercise

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

from sentence_transformers import SentenceTransformer
import faiss

In this exercise, you've been provided the title and abstract of 500 recent machine learning research papers posted on arXiv.org.

In [2]:
articles = pd.read_csv('../data/arxiv_papers.csv')
articles.head()

,title,abstract,url
0,GoT-R1: Unleashing Reasoning Capability of MLL...,Visual generation models have made remarkable ...,http://arxiv.org/abs/2505.17022v1
1,Delving into RL for Image Generation with CoT:...,Recent advancements underscore the significant...,http://arxiv.org/abs/2505.17017v1
2,Interactive Post-Training for Vision-Language-...,"We introduce RIPT-VLA, a simple and scalable r...",http://arxiv.org/abs/2505.17016v1
3,When Are Concepts Erased From Diffusion Models?,"Concept erasure, the ability to selectively pr...",http://arxiv.org/abs/2505.17013v1
4,Understanding Prompt Tuning and In-Context Lea...,Prompting is one of the main ways to adapt a p...,http://arxiv.org/abs/2505.17010v1


In [3]:
i = 0
print(f'Title: {articles.loc[i,"title"]}\n')
print(f'Text: {articles.loc[i,"abstract"]}')

Title: GoT-R1: Unleashing Reasoning Capability of MLLM for Visual Generation with Reinforcement Learning

Text: Visual generation models have made remarkable progress in creating realistic
images from text prompts, yet struggle with complex prompts that specify
multiple objects with precise spatial relationships and attributes. Effective
handling of such prompts requires explicit reasoning about the semantic content
and spatial layout. We present GoT-R1, a framework that applies reinforcement
learning to enhance semantic-spatial reasoning in visual generation. Building
upon the Generation Chain-of-Thought approach, GoT-R1 enables models to
autonomously discover effective reasoning strategies beyond predefined
templates through carefully designed reinforcement learning. To achieve this,
we propose a dual-stage multi-dimensional reward framework that leverages MLLMs
to evaluate both the reasoning process and final output, enabling effective
supervision across the entire generation pipeli

Let's try out a variety of ways of vectorizing and searching for semantically-similar papers.

### Method 1: Bag of Words

Fit a CountVectorizer to the abstracts of the articles with all of the defaults.  Then vectorize the dataset using the fit vectorizer. 

In [25]:
vectorizer = CountVectorizer()

X_abstracts = vectorizer.fit_transform(articles['abstract'])

**Question:** How many dimensions do the embeddings have?

In [26]:
print(X_abstracts.shape)

(500, 7978)


Now, let's use the embeddings to look for similar articles to a search query.

Apply the vectorizer you fit earlier to this query string to get an embedding. 

**Hint:** You can't pass a string to a vectorizer, but you can pass a list containing a string.

In [32]:
query = "vector databases for retrieval augmented generation"

# Your code to transform the search query
Y_query = vectorizer.transform([query])

Now, we need to find the similarity between our query embedding and each vectorized article.

For this, you can use the [cosine similarity function from scikit-learn.](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html)

Calculate the similarity between the query embedding and each article embedding and save the result to a variable named `similarity_scores`.

In [33]:
Y_query

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 6 stored elements and shape (1, 7978)>

In [34]:
X_abstracts

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 59465 stored elements and shape (500, 7978)>

In [35]:
similarity_scores = cosine_similarity(X_abstracts, Y_query)
similarity_scores

array([[0.11020775],
       [0.08461622],
       [0.01780047],
       [0.04145133],
       [0.03806935],
       [0.0814463 ],
       [0.06593805],
       [0.03782347],
       [0.        ],
       [0.07059312],
       [0.03462717],
       [0.07968191],
       [0.03857584],
       [0.020646  ],
       [0.03197647],
       [0.01893206],
       [0.04279605],
       [0.03979361],
       [0.05059374],
       [0.01539738],
       [0.01800706],
       [0.0184995 ],
       [0.03533326],
       [0.        ],
       [0.01879115],
       [0.03510393],
       [0.12309149],
       [0.        ],
       [0.04543109],
       [0.09829464],
       [0.10193473],
       [0.062177  ],
       [0.02197935],
       [0.11200358],
       [0.05493732],
       [0.04188539],
       [0.03503922],
       [0.06100889],
       [0.09866713],
       [0.07699905],
       [0.02475369],
       [0.03209979],
       [0.05309942],
       [0.11826248],
       [0.        ],
       [0.02207554],
       [0.0402259 ],
       [0.097

Now, we need to find the most similar results. To help with this, we can use the [argsort function from numpy](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html), which will give the indices sorted by value. 

Use the argsort function to find the indices of the 5 most similar articles. Inspect their titles and abstracts. **Warning:** argsort sorts from smallest to largest.

In [50]:
x = similarity_scores
x.shape

(500, 1)

In [41]:
ind = np.argsort(x, axis=0)  # sorts along first axis (down)
ind
np.take_along_axis(x, ind, axis=0)  # same as np.sort(x, axis=0)

array([[0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.   

In [ ]:
# change dataframe into a list
flat_scores = similarity_scores.flatten()

In [51]:
flat_scores

array([0.11020775, 0.08461622, 0.01780047, 0.04145133, 0.03806935,
       0.0814463 , 0.06593805, 0.03782347, 0.        , 0.07059312,
       0.03462717, 0.07968191, 0.03857584, 0.020646  , 0.03197647,
       0.01893206, 0.04279605, 0.03979361, 0.05059374, 0.01539738,
       0.01800706, 0.0184995 , 0.03533326, 0.        , 0.01879115,
       0.03510393, 0.12309149, 0.        , 0.04543109, 0.09829464,
       0.10193473, 0.062177  , 0.02197935, 0.11200358, 0.05493732,
       0.04188539, 0.03503922, 0.06100889, 0.09866713, 0.07699905,
       0.02475369, 0.03209979, 0.05309942, 0.11826248, 0.        ,
       0.02207554, 0.0402259 , 0.09707137, 0.06233787, 0.06225728,
       0.05412659, 0.06384424, 0.        , 0.        , 0.01944039,
       0.1214167 , 0.0285133 , 0.09924856, 0.        , 0.        ,
       0.        , 0.06944891, 0.07385489, 0.0285831 , 0.06463956,
       0.02733833, 0.03914801, 0.0955637 , 0.0843274 , 0.02810497,
       0.04072315, 0.05878964, 0.05236631, 0.09014453, 0.03219

In [ ]:
# get the indices sorted from lowest score to highest score
sorted_indices = np.argsort(flat_scores)

In [52]:
sorted_indices

array([396,  98, 303, 374,  87, 298, 481, 393, 458, 173, 321, 176, 291,
       413, 189,  60,  59,  58, 287, 286, 281,  53,  99,  52, 104, 109,
       143, 142, 140, 331, 464, 335, 345, 152, 485, 156, 316, 121, 349,
       158, 159, 116, 115, 310, 113, 306, 110, 164, 198, 322, 246,  27,
       432,   8, 252,  23, 430, 235, 220, 263, 239, 257, 428, 256, 451,
       475, 268, 493, 422,  44, 446, 300, 431,  19, 102,  77, 131, 267,
       336, 466, 353, 424,  81, 119,   2,  20, 237, 151, 168,  21, 436,
       382,  24,  80,  15, 277, 472, 370, 496, 378, 371,  54, 416, 135,
       144, 293, 231,  13, 240,  82, 188, 325, 365, 375, 178, 123, 269,
        96,  32, 171, 219,  45, 366, 448, 226, 433, 377, 409, 108, 380,
       415, 292, 412, 319, 234, 199, 179, 174, 341,  40, 167, 327, 364,
       379,  94, 470, 149, 244, 297, 453, 282,  65, 348, 323, 362, 228,
       203,  69, 442, 192, 208,  56, 232,  63, 467, 258, 482, 202,  91,
       201, 182, 305, 183, 401,  76, 423, 288, 242, 177, 418,  1

In [ ]:
# Grab the last 5 elements (the highest scores) and reverse them
top_5_indices = sorted_indices[-5:][::-1]    # This code is used to reverse the list [::-1] 

> You can attach [::-1] to the end of any standard Python list, string, or NumPy array to completely reverse its order.

In [48]:
print("Top 5 Article Indices:", top_5_indices)
print("Top 5 Similarity Scores:", flat_scores[top_5_indices])

Top 5 Article Indices: [259 289  83 486 394]
Top 5 Similarity Scores: [0.26978155 0.18926175 0.18057878 0.17057206 0.15681251]


> okay so this code sorted_indices = np.argsort(flat_scores) is takng the similarity scores between our query and each article then telling us where each article lands in terms of similarity to the phrase this the last 5 indices in this list indicte the 5 most similair articles to our phrase?

Try using a tfidf vectorizer. How do the results compare?

In [ ]:
# fit tfid vecorizer
tfidf_vectorizer = TfidfVectorizer()
X_abstracts_tfidf = tfidf_vectorizer.fit_transform(articles['abstract'])

In [ ]:
# create and transform datatable object to fit against
query = "vector databases for retrieval augmented generation"
Y_query_tfidf = tfidf_vectorizer.transform([query])

In [ ]:
# calculate similairity scores and flatten data or turn dataframe into list
tfidf_scores = cosine_similarity(X_abstracts_tfidf, Y_query_tfidf).flatten()

In [ ]:
# Show top 5 similairities
top_5_tfidf_indices = np.argsort(tfidf_scores)[-5:][::-1]

In [53]:
print("TF-IDF Top 5 Indices:", top_5_tfidf_indices)
print("TF-IDF Top 5 Scores:", tfidf_scores[top_5_tfidf_indices])

Task was destroyed but it is pending!
task: <Task pending name='Task-443' coro=<_async_in_context.<locals>.run_in_context() done, defined at /opt/anaconda3/envs/sentence_transformers/lib/python3.13/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-444' coro=<Kernel.shell_main() running at /opt/anaconda3/envs/sentence_transformers/lib/python3.13/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /opt/anaconda3/envs/sentence_transformers/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py:563]>
/opt/anaconda3/envs/sentence_transformers/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:1245: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  for term, old_index in list(vocabulary.items()):
Task was destroyed but it is pending!
task: <Task pending name='Task-444' coro=<Kernel.shell_main() running at /opt/anaconda3/envs/sentence_transformers/lib/python3.13/site-package

TF-IDF Top 5 Indices: [259  83 289 233 486]
TF-IDF Top 5 Scores: [0.29961889 0.26498602 0.21030972 0.15562194 0.14238743]


### Method 2: Using a Pretrained Embedding Model

Now, let's compare how we do using the [all-MiniLM-L6-v2 embedding model](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2).

This will create a 384-dimensional dense embedding of each sentence.

In [54]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [55]:
sentences = ["This is an example sentence", "Each sentence is converted"]

embeddings = embedder.encode(sentences)
print(embeddings)

[[ 6.76568449e-02  6.34958670e-02  4.87130582e-02  7.93049559e-02
   3.74480449e-02  2.65281321e-03  3.93748991e-02 -7.09841680e-03
   5.93614578e-02  3.15370336e-02  6.00980595e-02 -5.29051758e-02
   4.06067595e-02 -2.59308200e-02  2.98427939e-02  1.12694665e-03
   7.35150054e-02 -5.03819995e-02 -1.22386634e-01  2.37028394e-02
   2.97264718e-02  4.24769074e-02  2.56338231e-02  1.99519028e-03
  -5.69190793e-02 -2.71598399e-02 -3.29035632e-02  6.60248548e-02
   1.19007140e-01 -4.58791479e-02 -7.26215318e-02 -3.25839296e-02
   5.23414314e-02  4.50553112e-02  8.25294573e-03  3.67023498e-02
  -1.39415534e-02  6.53919280e-02 -2.64272727e-02  2.06370431e-04
  -1.36643555e-02 -3.62809710e-02 -1.95043385e-02 -2.89738197e-02
   3.94270681e-02 -8.84090886e-02  2.62425561e-03  1.36713833e-02
   4.83062677e-02 -3.11565232e-02 -1.17329165e-01 -5.11690527e-02
  -8.85287374e-02 -2.18961760e-02  1.42986430e-02  4.44167741e-02
  -1.34814689e-02  7.43392184e-02  2.66382825e-02 -1.98762063e-02
   1.79190

Use this new embedder to vectorize the abstracts and then find the most similar to the query. How do the results compare to the other methods?

**Warning:** Creating embeddings for all of the articles may take a while.

In [62]:
abstracts_vectorized = embedder.encode(articles['abstract'].tolist())

In [73]:
query = "vector databases for retrieval augmented generation"
query_embedding = embedder.encode([query])
similarity_scores = cosine_similarity(query_embedding, abstracts_vectorized)
most_similar_articles = np.argsort(-similarity_scores)[0,:5]
for title, abstract in articles.loc[most_similar_articles, ['title', 'abstract']]. values:
    print(title)
    print(abstract)
    print("--------")

MIRB: Mathematical Information Retrieval Benchmark
Mathematical Information Retrieval (MIR) is the task of retrieving
information from mathematical documents and plays a key role in various
applications, including theorem search in mathematical libraries, answer
retrieval on math forums, and premise selection in automated theorem proving.
However, a unified benchmark for evaluating these diverse retrieval tasks has
been lacking. In this paper, we introduce MIRB (Mathematical Information
Retrieval Benchmark) to assess the MIR capabilities of retrieval models. MIRB
includes four tasks: semantic statement retrieval, question-answer retrieval,
premise retrieval, and formula retrieval, spanning a total of 12 datasets. We
evaluate 13 retrieval models on this benchmark and analyze the challenges
inherent to MIR. We hope that MIRB provides a comprehensive framework for
evaluating MIR systems and helps advance the development of more effective
retrieval models tailored to the mathematical domai

### FAISS

The [Faiss library](https://faiss.ai/index.html) is a library for efficient similarity search and clustering of dense vectors. It can be used to automate the process of finding the most similar abstracts.

If we want to use cosine similarity, we need to use the Inner Product. We also need to normalize our vectors so that they all have length 1.

Use the [normalize function](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.normalize.html) to normalize both the abstract vectors and the query vector.

In [75]:
abstracts_vectorized_normalized = normalize(abstracts_vectorized)
query_embedding_normalized = normalize(query_embedding)

In [76]:
(abstracts_vectorized_normalized**2).sum(axis = 1)

array([1.        , 1.        , 1.        , 1.        , 1.        ,
       1.0000001 , 1.        , 1.        , 1.0000001 , 1.0000001 ,
       1.        , 0.9999999 , 1.0000001 , 1.        , 1.0000001 ,
       1.0000001 , 1.0000002 , 1.        , 1.0000001 , 1.        ,
       1.        , 1.0000001 , 1.        , 1.        , 1.0000001 ,
       0.99999994, 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.0000002 , 1.        , 1.0000002 , 1.        ,
       0.9999999 , 1.0000002 , 1.        , 1.        , 1.        ,
       1.0000001 , 1.        , 0.9999999 , 1.0000001 , 0.99999994,
       1.0000002 , 1.0000001 , 1.0000001 , 1.0000001 , 1.0000001 ,
       1.        , 1.0000001 , 1.0000002 , 1.        , 0.9999999 ,
       0.9999999 , 1.        , 1.        , 1.0000001 , 1.        ,
       1.        , 1.        , 0.9999999 , 1.0000002 , 1.        ,
       1.        , 0.9999999 , 1.        , 1.0000001 , 1.        ,
       0.99999994, 1.0000001 , 1.        , 1.        , 1.     

Now, create an [IndexFlatIP object](https://github.com/facebookresearch/faiss/wiki/Faiss-indexes#summary-of-methods) that has dimensions equal to the dimensionality of your vectors. Then add your normalized abstract vectors.

Hint: You can mimic the example [here](https://github.com/facebookresearch/faiss/wiki/Getting-started#building-an-index-and-adding-the-vectors-to-it), but substitute in the IndexFlatIP class.

In [80]:
dimension = abstracts_vectorized.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(abstracts_vectorized_normalized)

Finally, use the [search function](https://github.com/facebookresearch/faiss/wiki/Getting-started#searching) on your index object to find the 5 most similar articles.

In [78]:
index.search(query_embedding_normalized, k=5)

(array([[0.47817045, 0.4554737 , 0.45146078, 0.43434614, 0.4144664 ]],
       dtype=float32),
 array([[259, 107, 233, 209, 247]]))

In [79]:
D, I = index.search(query_embedding_normalized, k=5)
most_similar_articles = I[0]

for title, abstract in articles.loc[most_similar_articles, ['title', 'abstract']].values:
    print(title)
    print(abstract)
    print("--------")

MIRB: Mathematical Information Retrieval Benchmark
Mathematical Information Retrieval (MIR) is the task of retrieving
information from mathematical documents and plays a key role in various
applications, including theorem search in mathematical libraries, answer
retrieval on math forums, and premise selection in automated theorem proving.
However, a unified benchmark for evaluating these diverse retrieval tasks has
been lacking. In this paper, we introduce MIRB (Mathematical Information
Retrieval Benchmark) to assess the MIR capabilities of retrieval models. MIRB
includes four tasks: semantic statement retrieval, question-answer retrieval,
premise retrieval, and formula retrieval, spanning a total of 12 datasets. We
evaluate 13 retrieval models on this benchmark and analyze the challenges
inherent to MIR. We hope that MIRB provides a comprehensive framework for
evaluating MIR systems and helps advance the development of more effective
retrieval models tailored to the mathematical domai